In [2]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework06') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/14 20:53:10 WARN Utils: Your hostname, omtest-Virtual-Machine, resolves to a loopback address: 127.0.1.1; using 172.24.50.174 instead (on interface eth0)
26/03/14 20:53:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 20:53:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-14 20:53:21--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.1.56, 18.239.1.74, 18.239.1.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.1.56|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M  2.69MB/s    in 32s     

2026-03-14 20:53:54 (2.10 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [9]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [10]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [11]:
#Question 2:
df = df.repartition(4)

In [13]:
df.write.parquet("partitions/")

In [14]:
df = spark.read.parquet("partitions/")

In [15]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [16]:
from pyspark.sql import functions as F

In [18]:
df.createOrReplaceTempView("yellow_tripdata")

In [20]:
#Question 3: 
spark.sql("""
SELECT
    count(*)
FROM
    yellow_tripdata
WHERE
   tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime < '2025-11-16 00:00:00'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [23]:
#Question 4:
spark.sql("""
SELECT 
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) as max_hours
FROM yellow_tripdata;
""").show()

+-----------------+
|        max_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [24]:
#Question 6:
!wget -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-14 21:23:37--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.1.70, 18.239.1.74, 18.239.1.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.1.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-14 21:23:37 (261 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [46]:
df_zone = spark.read.option("header","true").csv('taxi_zone_lookup.csv')

In [47]:
df_zone.schema

StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [49]:
result = (
    df.alias("y")
    .join(
        df_zone.alias("z"), 
        F.col("y.PULocationID") == F.col("z.LocationID"), 
        "inner"
    )
    # Grouping by both ID and Borough to ensure Borough stays in the output
    .groupBy("z.Zone", "y.PULocationID") 
    .agg(F.count("*").alias("frequency"))
    .orderBy(F.col("frequency").asc())
)

result.show()

[Stage 32:>                                                         (0 + 2) / 2]

+--------------------+------------+---------+
|                Zone|PULocationID|frequency|
+--------------------+------------+---------+
|Eltingville/Annad...|          84|        1|
|       Arden Heights|           5|        1|
|Governor's Island...|         105|        1|
|       Port Richmond|         187|        3|
|       Rikers Island|         199|        4|
|   Rossville/Woodrow|         204|        4|
|         Great Kills|         109|        4|
| Green-Wood Cemetery|         111|        4|
|         Jamaica Bay|           2|        5|
|         Westerleigh|         251|       12|
|       West Brighton|         245|       14|
|             Oakwood|         176|       14|
|New Dorp/Midland ...|         172|       14|
|        Crotona Park|          59|       14|
|       Willets Point|         253|       15|
|Breezy Point/Fort...|          27|       16|
|Saint George/New ...|         206|       17|
|       Broad Channel|          30|       18|
|     Mariners Harbor|         156